In [ ]:
import numpy as np
from tensorflow.keras.layers import Dense, SimpleRNN
from tensorflow.keras.models import Sequential
import tensorflow as tf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/Colab Notebooks/cuento_EAPoe_1.txt'

texto = open(path, 'r', encoding='utf8').read()
caracteres = list(set(texto))
tam_vocab = len(caracteres)

print(caracteres)
print(tam_vocab)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['x', 'L', 'Ú', ')', '5', 'D', 'o', 'N', 'M', 'ü', 'b', 'Á', '1', 'O', 'é', ']', 'z', ',', '}', 'l', 'd', '6', 'V', 'A', 'K', '¶', 'F', 'R', '(', 'â', 'Í', '{', 'c', 'Ó', '9', 'I', 'J', '4', '!', 'p', 's', '7', '.', 'i', 'f', 'S', 'e', 'h', 'á', '"', 'ö', 'ú', "'", '_', 'Q', 'C', '\n', ' ', 'É', ':', 'H', 'B', 'Ö', '2', 'r', 'm', 'a', 'E', 't', '8', 'j', '3', '0', 'u', 'k', 'q', 'W', 'X', 'Y', 'v', 'T', '[', 'g', '¡', 'ô', '-', 'w', 'ñ', 'U', ';', 'ó', 'y', 'n', '?', 'P', '*', 'í', 'æ', 'G', '¿']
100


In [ ]:
#Diccionarios para convertir texto a un indice numerico y viceversa
caracteres_ix = {c:i for i,c in enumerate(caracteres)}
ix_caracteres = {i:c for i,c  in enumerate(caracteres)}
print(caracteres_ix)
print(ix_caracteres)

{'x': 0, 'L': 1, 'Ú': 2, ')': 3, '5': 4, 'D': 5, 'o': 6, 'N': 7, 'M': 8, 'ü': 9, 'b': 10, 'Á': 11, '1': 12, 'O': 13, 'é': 14, ']': 15, 'z': 16, ',': 17, '}': 18, 'l': 19, 'd': 20, '6': 21, 'V': 22, 'A': 23, 'K': 24, '¶': 25, 'F': 26, 'R': 27, '(': 28, 'â': 29, 'Í': 30, '{': 31, 'c': 32, 'Ó': 33, '9': 34, 'I': 35, 'J': 36, '4': 37, '!': 38, 'p': 39, 's': 40, '7': 41, '.': 42, 'i': 43, 'f': 44, 'S': 45, 'e': 46, 'h': 47, 'á': 48, '"': 49, 'ö': 50, 'ú': 51, "'": 52, '_': 53, 'Q': 54, 'C': 55, '\n': 56, ' ': 57, 'É': 58, ':': 59, 'H': 60, 'B': 61, 'Ö': 62, '2': 63, 'r': 64, 'm': 65, 'a': 66, 'E': 67, 't': 68, '8': 69, 'j': 70, '3': 71, '0': 72, 'u': 73, 'k': 74, 'q': 75, 'W': 76, 'X': 77, 'Y': 78, 'v': 79, 'T': 80, '[': 81, 'g': 82, '¡': 83, 'ô': 84, '-': 85, 'w': 86, 'ñ': 87, 'U': 88, ';': 89, 'ó': 90, 'y': 91, 'n': 92, '?': 93, 'P': 94, '*': 95, 'í': 96, 'æ': 97, 'G': 98, '¿': 99}
{0: 'x', 1: 'L', 2: 'Ú', 3: ')', 4: '5', 5: 'D', 6: 'o', 7: 'N', 8: 'M', 9: 'ü', 10: 'b', 11: 'Á', 12: '1', 

In [ ]:
from IPython.utils import text
long_seq = 10 #Longitud secuencia de entrada
texto_in, texto_out = [], []

for i in range(0, len(texto)-long_seq, 1):
  texto_in.append(texto[i:i+long_seq])
  texto_out.append(texto[i+long_seq])

print(texto_in[0])
print(texto_out[0])
print(texto_in[1])
print(texto_out[1])

EL BARRIL 
D
L BARRIL D
E


Conjunto de entrenamiento se debe convertie a one-hot tanto salida como entrada

In [ ]:
X = np.zeros((len(texto_in), long_seq, tam_vocab))#crea tensor con ceros que serian las entradas del modelo del tamaño del vocab = 100
Y = np.zeros((len(texto_in), tam_vocab))#crea tensor con ceros que serian las salidas del modelo

for i, entrada in enumerate(texto_in):
  for j, caracter in enumerate(entrada):
    X[i, j, caracteres_ix[caracter]] = 1
    Y[i, caracteres_ix[texto_out[i]]] = 1

print(X.shape)
print(Y.shape)
print(X[0,0,:])#primer vector codificado del primer caracter
print(Y[0,:])

(331185, 10, 100)
(331185, 100)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


331185 ejemplos
10 secuencias
100 caracteres

**Crear el modelo**

In [ ]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

N_neuronas = 256

modelo_RNN = Sequential()
modelo_RNN.add(SimpleRNN(N_neuronas, input_shape=(long_seq, tam_vocab)))
modelo_RNN.add(Dense(tam_vocab, activation='softmax'))

modelo_RNN.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 256)            │        91,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        25,700 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,092 (457.39 KB)

 Trainable params: 117,092 (457.39 KB)

 Non-trainable params: 0 (0.00 B)

Funcion para generar el texto

In [ ]:
def generar_texto(modelo_RNN, L_OUT_SEC):
  ix_test = np.random.randint(len(texto_in))
  caracter_test = texto_in[ix_test]
  print(caracter_test, end='')

  for i in range(L_OUT_SEC):
    X_test = np.zeros((1, long_seq, tam_vocab)) #crear tensor vacio para representar entrada al modelo
    for j, caracter in enumerate(caracter_test):
      X_test [0, j, caracteres_ix[caracter]] = 1

      #Predecir el siguiente caracter usando el modelo entrenado
    pred = modelo_RNN.predict(X_test, verbose=0)[0]

    y_pred = ix_caracteres[np.argmax(pred)]

    print(y_pred, end='')

    caracter_test = caracter_test[1:] + y_pred #Va corriendo e imprimiendo el siguiente
  print()

In [ ]:
modelo_RNN.compile(loss='categorical_crossentropy', optimizer='rmsprop')
Nepocas = 10
Batch_tam = 64 #En cuantas partes divido
L_OUT_SEC = 20

print('_'*100)
print("Antes del entrenamiento")
generar_texto(modelo_RNN, L_OUT_SEC)

for it in range(Nepocas):
  modelo_RNN.fit(X, Y, batch_size=Batch_tam, epochs=1, verbose=2)
  print('_'*100)
  print(f"Epoca {it+1}")
  generar_texto(modelo_RNN, L_OUT_SEC)

____________________________________________________________________________________________________
Antes del entrenamiento
ido dos estaba al fin alguna v
5175/5175 - 57s - 11ms/step - loss: 1.6523
____________________________________________________________________________________________________
Epoca 1
panaye, en verdad, en con los 
5175/5175 - 61s - 12ms/step - loss: 1.6402
____________________________________________________________________________________________________
Epoca 2
, los cuales pero de la puerta
5175/5175 - 60s - 12ms/step - loss: 1.6278
____________________________________________________________________________________________________
Epoca 3
 manifiestado de la casa de la
5175/5175 - 60s - 12ms/step - loss: 1.6200
____________________________________________________________________________________________________
Epoca 4
os
cilíndridad en rado se desc
5175/5175 - 60s - 12ms/step - loss: 1.6126
_____________________________________________________________________